# ML-04 Data Contract: Refresh and Content Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oumaklaus/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

A data contract for my lane, checked against the gated warehouse (`hf://datasets/FlyRank/internship-warehouse`), not the starter CSV. Four parts, in order:

1. The contract in plain words, five answers.
2. Three facts, three small queries, on one mid panel month (`month=2026-03`).
3. Five features, each with why it is known when the decision is made, then the leak trap.
4. One honest limitation of the slice.

Plain words, honest numbers. Every claim below has a query under it.

## 1. The contract in plain words

My lane is Refresh, or Content Opportunity Scoring. I rank pages that look like they are about to lose search traffic, so a person can review the ones near the top and decide whether to refresh them.

Five answers.

1. **What one row means.** One row is one page in one month. I collapse each page's daily rows in the panel month into a single record keyed by `content_hash_id`, because the decision, should someone refresh this page, is made per page, not per day.

2. **Which tables.** Only `fact_content_daily_performance`, the daily Search Console and GA4 fact, one row per page per day. That single table carries both the features and the label. `dim_clients` and `dim_content` exist for context, who owns a page and when its history starts, but I do not pull them into the scoring frame.

3. **Which time window.** Features come from March 2026 (`month=2026-03`). The label comes from the following month, April 2026 (`month=2026-04`). The two windows never overlap, so nothing I predict from can contain the answer.

4. **What I rank, the label as a proxy.** A page counts as a decline (label 1) when its April search impressions fall more than 20 percent below March, that is `april_impr < 0.8 * march_impr`. This stands in for needs a refresh, since I cannot see an editor's real intent and a sharp forward drop in impressions is the closest thing I can observe. The label lives entirely in April and is never a feature.

5. **One thing I deliberately exclude.** Pages where Search Console has no data in March. Their `gsc_data_available` flag is not TRUE, they carry no impression, click, or position signal to rank on, and the flag is three valued (TRUE, FALSE, or NULL). I keep the slice to `gsc_data_available IS TRUE`. This is the limitation I return to at the end.

In [1]:
# Setup: connect DuckDB to the gated warehouse.
# The token is read from the environment, Colab Secrets, or a prompt.
# It is NEVER written into this notebook (public repo).
import os, duckdb, numpy as np, pandas as pd

def _hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata  # Colab: Secrets -> add HF_TOKEN
        return userdata.get("HF_TOKEN")
    except Exception:
        import getpass
        return getpass.getpass("HF read token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{_hf_token()}')")

REL  = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MAR  = f"read_parquet('{FACT}/month=2026-03/*.parquet')"   # panel month = features
APR  = f"read_parquet('{FACT}/month=2026-04/*.parquet')"   # next month  = label

def q(sql):
    return con.sql(sql).df()

print("connected; panel month = 2026-03, label month = 2026-04")

HF read token: ··········
connected; panel month = 2026-03, label month = 2026-04


## 2. Three facts, three small queries

Everything above is a claim until a query backs it. Here are exactly three, all on the panel month (`month=2026-03`), each small enough to touch one month's partition.

* **Q1, the grain.** Is `content_hash_id` unique within a day, so the fact really is one row per page per day?
* **Q2, my slice.** After keeping only `gsc_data_available IS TRUE`, how many distinct pages do I have and what date span do they cover?
* **Q3, availability.** Of all March daily rows, how many survive the `IS TRUE` filter and how many do not?

Three queries, no more. I iterate on this one month. A full table scan is a last resort, since the fact holds about 79 million rows and hammering it returns HTTP 429.

In [2]:
# Exactly three small queries on the panel month (month=2026-03).

# Q1 - the grain: is the fact one row per page per day?
q1 = q(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS c
    FROM {MAR}
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""")
print("Q1 grain - rows where a page repeats within a day (want 0):", len(q1))

# Q2 - my slice: keep gsc_data_available IS TRUE, then count distinct pages
#      and read the date span they cover.
q2 = q(f"""
    SELECT COUNT(DISTINCT content_hash_id) AS pages,
           MIN(report_date) AS span_start,
           MAX(report_date) AS span_end
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
""")
print("\nQ2 slice - pages + date span:")
print(q2.to_string(index=False))

# Q3 - availability: of all March daily rows, how many survive IS TRUE (and how many don't)?
q3 = q(f"""
    SELECT COUNT(*) AS total_daily_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)     AS is_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS is_not_true
    FROM {MAR}
""")
print("\nQ3 availability - daily rows split by the three valued flag:")
print(q3.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Q1 grain - rows where a page repeats within a day (want 0): 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Q2 slice - pages + date span:
 pages span_start   span_end
176738 2026-03-01 2026-03-31

Q3 availability - daily rows split by the three valued flag:
 total_daily_rows  is_true  is_not_true
          9841378  3611061      6230317


## 3. Five features, then the trap

Same panel month, same slice. I build one small feature frame at the page level, five columns, each read only from March so it is known at the moment the refresh decision is made (end of March, before April starts). Then I attach the April label and run a quick score. Finally I add one column that peeks at April on purpose, watch the score jump toward perfect, and delete it.

The five features, and why each is known at the decision moment:

1. `mar_impr`, total March Search Console impressions. Known at end of March because it only sums days on or before March 31, before the April label window opens.
2. `mar_clicks`, total March clicks. Same reason, measured inside the feature month.
3. `ctr`, March clicks over impressions times 100. Both parts are March only, so it is fixed the moment March closes.
4. `avg_position`, impression weighted mean search position across March. Read straight from March rows, no future data touches it.
5. `days_with_impr`, the count of March days the page drew at least one impression. A within month activity count, complete once March ends.

In [3]:
# Build the page level feature frame from March, keep only gsc_data_available IS TRUE.
feat = q(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions)                                      AS mar_impr,
           SUM(gsc_clicks)                                           AS mar_clicks,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
           SUM(gsc_sum_position)   / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)               AS days_with_impr
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""")
print("feature frame:", feat.shape)
print(feat.head().to_string(index=False))

# Attach the April label: a page declines if April impressions fall more than 20% below March.
label = q(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS apr_impr
    FROM {APR}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""")
frame = feat.merge(label, on="content_hash_id", how="left")
frame["apr_impr"] = frame["apr_impr"].fillna(0)
frame["label"] = (frame["apr_impr"] < 0.8 * frame["mar_impr"]).astype(int)
print(f"\nlabeled pages: {len(frame):,}   decline rate: {frame['label'].mean():.3f}   "
      f"({int(frame['label'].sum()):,} of {len(frame):,})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature frame: (176738, 6)
         content_hash_id  mar_impr  mar_clicks      ctr  avg_position  days_with_impr
content_7a105f548d9c6916    6523.0         7.0 0.107313      6.893301              31
content_a3ea9792f793ec72     453.0         0.0 0.000000      3.214128              31
content_36c36abc7650d7af    5630.0         6.0 0.106572      6.535346              31
content_a7da352b73b02668    4944.0        13.0 0.262945      7.435680              31
content_1855a661b4d36130     429.0         1.0 0.233100      3.871795              31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


labeled pages: 176,738   decline rate: 0.532   (94,000 of 176,738)


### The trap, done on purpose

The five features above give an honest, unglamorous score. Now I add one more column, `leaky_apr_ratio`, which is April impressions divided by March impressions. April is the label window, so this column already knows the answer. Watch the score jump toward perfect, then delete the column and keep the honest number. This is the leakage lesson from notebook 02, run on real warehouse data.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

HONEST = ["mar_impr", "mar_clicks", "ctr", "avg_position", "days_with_impr"]
y = frame["label"].values

def quick_score(cols):
    X = frame[cols].fillna(0.0).values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), ytr)
    p = clf.predict_proba(sc.transform(Xte))[:, 1]
    top = np.argsort(-p)[:50]
    return roc_auc_score(yte, p), yte[top].mean()

# 1) honest score, the five features only
h_auc, h_p = quick_score(HONEST)
print(f"honest  : AUC {h_auc:.3f}   precision@50 {h_p:.3f}")

# 2) add the leak on purpose
frame["leaky_apr_ratio"] = frame["apr_impr"] / frame["mar_impr"]
l_auc, l_p = quick_score(HONEST + ["leaky_apr_ratio"])
print(f"leaky   : AUC {l_auc:.3f}   precision@50 {l_p:.3f}   <- looks perfect, and it is a lie")

# 3) delete the leak, keep the honest number
frame = frame.drop(columns=["leaky_apr_ratio"])
print(f"\nkept    : AUC {h_auc:.3f} is the honest score. The leak is gone.")

honest  : AUC 0.589   precision@50 0.760
leaky   : AUC 0.999   precision@50 1.000   <- looks perfect, and it is a lie

kept    : AUC 0.589 is the honest score. The leak is gone.


## 4. One honest limitation

The slice is Search Console pages only. In March, `IS TRUE` kept 3,611,061 of 9,841,378 daily rows; the other 6,230,317 did not survive it (see Q3 above). That is roughly two in three daily rows dropped before I score anything.

So the queue is blind to every page FlyRank cannot yet measure in Search Console, whether the client has no GSC connection or the page simply drew no search data that month. A page can be quietly losing traffic and never appear in my ranking. The score is only honest about the pages the warehouse can see, and I should say so plainly rather than let the queue read as if it covered everything.

In [5]:
# Cache the verified numbers for the repo (best effort; harmless if the path differs in Colab).
import json
metrics = {
    "task": "ML-04 w03_data_contract",
    "lane": "Refresh / Content Opportunity Scoring",
    "warehouse": REL,
    "panel_month": "2026-03",
    "label_month": "2026-04",
    "grain": "one page (content_hash_id) per month, gsc_data_available IS TRUE",
    "verified": {
        "march_daily_rows": int(q3["total_daily_rows"][0]),
        "gsc_available_is_true_rows": int(q3["is_true"][0]),
        "gsc_available_is_not_true_rows": int(q3["is_not_true"][0]),
        "slice_pages": int(q2["pages"][0]),
        "march_span": [str(q2["span_start"][0].date()), str(q2["span_end"][0].date())],
        "labeled_pages": int(len(frame)),
        "label_pos_rate": round(float(frame["label"].mean()), 4),
        "label_n_pos": int(frame["label"].sum()),
        "label_rule": "april_impr < 0.8 * march_impr (more than 20% forward decline)",
        "median_days_with_impr": int(frame["days_with_impr"].median()),
    },
    "quick_score": {
        "features_honest": HONEST,
        "leak_column": "leaky_apr_ratio = apr_impr / mar_impr",
        "honest": {"auc": round(float(h_auc), 3), "p_at_50": round(float(h_p), 3)},
        "leaky": {"auc": round(float(l_auc), 3), "p_at_50": round(float(l_p), 3)},
        "model": "LogisticRegression(max_iter=1000) on StandardScaler features",
        "split": "train_test_split test_size=0.3 stratify=y random_state=42",
    },
    "seed": 42,
    "limitation": (
        "Slice is GSC-available pages only; pages with gsc_data_available IS NOT TRUE "
        "(6,230,317 of 9,841,378 March daily rows) are excluded, so the queue is blind "
        "to pages FlyRank cannot yet measure in Search Console."
    ),
}
try:
    os.makedirs("work/outputs", exist_ok=True)
    with open("work/outputs/w03_data_contract_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print("wrote work/outputs/w03_data_contract_metrics.json")
except Exception as e:
    print("metrics not written:", e)
print(json.dumps(metrics["verified"], indent=2))

wrote work/outputs/w03_data_contract_metrics.json
{
  "march_daily_rows": 9841378,
  "gsc_available_is_true_rows": 3611061,
  "gsc_available_is_not_true_rows": 6230317,
  "slice_pages": 176738,
  "march_span": [
    "2026-03-01",
    "2026-03-31"
  ],
  "labeled_pages": 176738,
  "label_pos_rate": 0.5319,
  "label_n_pos": 94000,
  "label_rule": "april_impr < 0.8 * march_impr (more than 20% forward decline)",
  "median_days_with_impr": 26
}


## 5. Self-check

Before you submit, confirm each line honestly:

- [x] The contract has five plain answers, and the three queries run with their outputs visible (availability checked with `IS TRUE`).
- [x] Five features, each with a line saying when it is known, plus the deliberate leak shown and then removed.
- [x] One named limitation of the slice, backed by the Q3 numbers.
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all).
- [x] No client names, URLs, or private queries anywhere; IDs stay pseudonymized.
- [x] Committed under `work/notebooks/`; then submit the repo URL on the card. Done.